In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import rasterio

drive_data_path = "/content/gdrive/MyDrive/colab_data"
os.makedirs(drive_data_path, exist_ok=True)
# def weighted_sce(y_true, y_pred):
#     y_true = tf.cast(y_true, tf.int32)
#     weights = tf.gather(class_weights, y_true)
#     loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
#     return loss * weights

# Load trained model
class_weights = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)  # example
def weighted_sce(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    weights = tf.gather(class_weights, y_true)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return loss * weights
model_path = os.path.join(drive_data_path,"model.keras")
model = tf.keras.models.load_model(model_path,custom_objects={"weighted_sce": weighted_sce})

# Load DEM
dem_path = os.path.join(drive_data_path, "Gale_Creater_dem.tif")

with rasterio.open(dem_path) as src:
    dem = src.read(1)

print("DEM shape:", dem.shape)

DEM shape: (198, 405)


In [5]:
dem = dem.astype(np.float32)

dem = (dem - np.min(dem)) / (np.max(dem) - np.min(dem))

In [15]:
PATCH_SIZE = 128
STRIDE = 16

In [16]:
height, width = dem.shape

prediction_map = np.zeros((height, width))
count_map = np.zeros((height, width))

Slide Window Across DEM

In [18]:
for i in range(0, height - PATCH_SIZE, STRIDE):
    for j in range(0, width - PATCH_SIZE, STRIDE):

        patch_dem = dem[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        patch_slope = slope[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        patch_rough = roughness[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        patch = np.stack([patch_dem, patch_slope, patch_rough], axis=-1)  # (128,128,3)
        patch = np.expand_dims(patch, axis=0)  # (1,128,128,3)
        pred = model.predict(patch, verbose=0)[0]

        prediction_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += pred
        count_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += 1

NameError: name 'slope' is not defined

In [ ]:
prediction_map = prediction_map / count_map

In [7]:
plt.figure(figsize=(12,6))
plt.imshow(prediction_map, cmap="viridis")
plt.colorbar(label="Landing Safety Probability")
plt.title("Mars Landing Safety Map - Elysium Planitia")
plt.show()

NameError: name 'prediction_map' is not defined

<Figure size 1200x600 with 0 Axes>